In [19]:
import os
from glob import glob
import numpy as np
import pyproj
from osgeo import osr, gdal
import rasterio
from rasterio.merge import merge

import spectral_util.spec_io as spec_io
import spectral_util.mosaic as mosaic

os.chdir('/store/carroll/col/data/2025/')

In [67]:
domain = 'UPTA'

input_file_list = f'mosaic/file_lists/top_priority_obs_{domain}.txt'
input_files = [x.strip() for x in open(input_file_list, 'r').readlines()]
tile_fps = [x for x in glob(f'mosaic/{domain}_2025_mosaic_glt_3*.tif') if 'aligned' not in x]

output_epsg = 32613
proj = pyproj.Proj(f"epsg:{output_epsg}")

x_resolution = 1
y_resolution = -1

In [68]:
# get ul, lr
ul_lr = mosaic.get_ul_lr_from_files(input_files, get_resolution=False)
# convert to output epsg
ul = proj(ul_lr[0], ul_lr[1])
lr = proj(ul_lr[2], ul_lr[3])
ul_lr = [ul[0], ul[1], lr[0], lr[1]]
print(ul_lr)

gproj = osr.SpatialReference()
gproj.ImportFromEPSG(int(output_epsg))
wkt = gproj.ExportToWkt()
proj = pyproj.Proj(f"epsg:{output_epsg}")

[343823.8206059444, 4325896.563726522, 362348.6232427064, 4300357.422358013]


In [69]:
# construct full mosaic grid per build_obs_nc
trans = [ul_lr[0] - x_resolution/2., x_resolution, 0, 
         ul_lr[1] - y_resolution/2., 0, y_resolution]
meta = spec_io.GenericGeoMetadata(['GLT X', 'GLT Y', 'File Index', 'OBS val'], 
                                  projection=wkt, 
                                  geotransform=trans, 
                                  pre_orthod=True, 
                                  nodata_value=0)

glt = np.ones(( int(np.ceil((ul_lr[3] - ul_lr[1]) / y_resolution)), 
                 int(np.ceil((ul_lr[2] - ul_lr[0]) / x_resolution)),
                 3), dtype=np.int32)

print(glt.shape)

output_file = f'mosaic/{domain}_reference_grid.tif'
spec_io.write_cog(output_file, glt, meta, nodata_value=0)

(25540, 18525, 3)


In [70]:
# warp each tile back to the original glt grid

ref_fp = f'mosaic/{domain}_reference_grid.tif'

ref = gdal.Open(ref_fp)
gt = ref.GetGeoTransform()
proj = ref.GetProjection()

xres = gt[1]
yres = abs(gt[5])

ref_ul_x = gt[0]
ref_ul_y = gt[3]
ref_lr_x = ref_ul_x + ref.RasterXSize * xres
ref_lr_y = ref_ul_y - ref.RasterYSize * yres

resample_alg = gdal.GRA_NearestNeighbour

warp_opts = gdal.WarpOptions(
    format="GTiff",
    dstSRS=proj,
    xRes=xres,
    yRes=yres,
    outputBounds=(ref_ul_x, ref_lr_y, ref_lr_x, ref_ul_y),
    targetAlignedPixels=True,
    resampleAlg=resample_alg,
    multithread=True,
    creationOptions=["TILED=YES", "COMPRESS=DEFLATE", "BIGTIFF=YES"],
)

aligned = []
for fp in tile_fps:
    fp_out = fp.replace('.tif', '_aligned.tif')
    gdal.Warp(str(fp_out), str(fp), options=warp_opts)
    aligned.append(fp_out)

In [84]:
aligned

['mosaic/UPTA_2025_mosaic_glt_343818.8206059444_4315685.907179118_347533.7811332968_4310568.078905417_aligned.tif',
 'mosaic/UPTA_2025_mosaic_glt_343818.8206059444_4320793.735452821_347533.7811332968_4315675.907179118_aligned.tif',
 'mosaic/UPTA_2025_mosaic_glt_347523.7811332968_4315685.907179118_351238.7416606492_4310568.078905417_aligned.tif',
 'mosaic/UPTA_2025_mosaic_glt_347523.7811332968_4320793.735452821_351238.7416606492_4315675.907179118_aligned.tif',
 'mosaic/UPTA_2025_mosaic_glt_347523.7811332968_4325901.563726522_351238.7416606492_4320783.735452821_aligned.tif',
 'mosaic/UPTA_2025_mosaic_glt_351228.7416606492_4310578.078905417_354943.7021880016_4305460.250631714_aligned.tif',
 'mosaic/UPTA_2025_mosaic_glt_351228.7416606492_4315685.907179118_354943.7021880016_4310568.078905417_aligned.tif',
 'mosaic/UPTA_2025_mosaic_glt_351228.7416606492_4320793.735452821_354943.7021880016_4315675.907179118_aligned.tif',
 'mosaic/UPTA_2025_mosaic_glt_351228.7416606492_4325901.563726522_354943

In [100]:
# Build final mosaic out of aligned tiles
nodata=0

vrt_fp = "mosaic/mosaic.vrt"
vrt_opts = gdal.BuildVRTOptions(
    srcNodata=nodata,   # treat src nodata as nodata
    VRTNodata=nodata,   # set the VRT nodata
)
gdal.BuildVRT(str(vrt_fp), [str(p) for p in aligned], options=vrt_opts)


fp_out = f'mosaic/{domain}_mosaic_glt_2025_vrt.tif'
translate_opts = gdal.TranslateOptions(
    noData=nodata,
    creationOptions=["TILED=YES", "COMPRESS=DEFLATE", "BIGTIFF=YES"],
)
gdal.Translate(
    fp_out,
    vrt_fp,
    options=translate_opts
)

<osgeo.gdal.Dataset; proxy of <Swig Object of type 'GDALDatasetShadow *' at 0x7f6f34a51440> >

In [101]:
print('done')

done


In [74]:
def info(fp):
    ds = gdal.Open(str(fp))
    gt = ds.GetGeoTransform()
    xres, yres = gt[1], abs(gt[5])
    xmin = gt[0]
    ymax = gt[3]
    xmax = xmin + ds.RasterXSize * xres
    ymin = ymax - ds.RasterYSize * yres
    return {
        "fp": str(fp),
        "size": (ds.RasterXSize, ds.RasterYSize),
        "res": (xres, yres),
        "bounds": (xmin, ymin, xmax, ymax),
        "proj_wkt_len": len(ds.GetProjection() or ""),
        "bands": ds.RasterCount,
        "dtype": ds.GetRasterBand(1).DataType,
        "nodata": ds.GetRasterBand(1).GetNoDataValue(),
    }


In [81]:
info(ref_fp)

{'fp': 'mosaic/UPTA_reference_grid.tif',
 'size': (18525, 25540),
 'res': (1.0, 1.0),
 'bounds': (343823.3206059444,
  4300357.063726522,
  362348.3206059444,
  4325897.063726522),
 'proj_wkt_len': 599,
 'bands': 3,
 'dtype': 5,
 'nodata': 0.0}

In [83]:
info(aligned[0])

{'fp': 'mosaic/UPTA_2025_mosaic_glt_343818.8206059444_4315685.907179118_347533.7811332968_4310568.078905417_aligned.tif',
 'size': (18526, 25541),
 'res': (1.0, 1.0),
 'bounds': (343823.0, 4300357.0, 362349.0, 4325898.0),
 'proj_wkt_len': 599,
 'bands': 3,
 'dtype': 5,
 'nodata': 0.0}

In [82]:
info('mosaic/UPTA_2025_mosaic_glt_354933.7021880016_4305470.250631714_358648.662715354_4300352.422358013_aligned.tif')

{'fp': 'mosaic/UPTA_2025_mosaic_glt_354933.7021880016_4305470.250631714_358648.662715354_4300352.422358013_aligned.tif',
 'size': (18526, 25541),
 'res': (1.0, 1.0),
 'bounds': (343823.0, 4300357.0, 362349.0, 4325898.0),
 'proj_wkt_len': 599,
 'bands': 3,
 'dtype': 5,
 'nodata': 0.0}

In [97]:


ds = [rasterio.open(fp) for fp in aligned]

mosaic, transform = merge(ds, method='first', nodata=nodata)

In [98]:
out_meta = ds[0].meta.copy()
out_meta.update({
    'height': mosaic.shape[1],
    'width': mosaic.shape[2],
    'transform': transform,
    'compress': 'DEFLATE',
    'tiled': True,
    'BIGTIFF': 'YES',
})

fp_out = f'mosaic/{domain}_mosaic_glt_2025.tif'

with rasterio.open(fp_out, 'w', **out_meta) as dst:
    dst.write(mosaic)

for ds in ds:
    ds.close()

In [99]:
out_meta

{'driver': 'GTiff',
 'dtype': 'int32',
 'nodata': 0.0,
 'width': 18526,
 'height': 25541,
 'count': 3,
 'crs': CRS.from_wkt('PROJCS["WGS 84 / UTM zone 13N",GEOGCS["WGS 84",DATUM["WGS_1984",SPHEROID["WGS 84",6378137,298.257223563,AUTHORITY["EPSG","7030"]],AUTHORITY["EPSG","6326"]],PRIMEM["Greenwich",0,AUTHORITY["EPSG","8901"]],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AUTHORITY["EPSG","4326"]],PROJECTION["Transverse_Mercator"],PARAMETER["latitude_of_origin",0],PARAMETER["central_meridian",-105],PARAMETER["scale_factor",0.9996],PARAMETER["false_easting",500000],PARAMETER["false_northing",0],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH],AUTHORITY["EPSG","32613"]]'),
 'transform': Affine(1.0, 0.0, 343823.0,
        0.0, -1.0, 4325898.0),
 'compress': 'DEFLATE',
 'tiled': True,
 'BIGTIFF': 'YES'}